# 🎯 Notebook 14: The Institutional Roadmap — From 55% to 60%+

---

**Author:** Tuhin Bhattacharya  
**Date:** February 2025  
**Series:** Tata Motors Deep Dive (14 of 15)

## Objective

In Notebook 08 we hit a ~ **55% accuracy ceiling**. This notebook answers:
1. **Why 55% is actually powerful** — the casino analogy
2. **How institutional quants break past 55%** — the 4-Level Upgrade Roadmap
3. **Meta-Labeling, Triple-Barrier, Order-Book, Alternative Data** — each level demonstrated with real numbers
4. **Summary Action Plan** — what to do RIGHT NOW to make 55% more profitable

> *"In the casino, the house edge is only 52%. That 2% is enough to build Las Vegas."*


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
PROCESSED_DIR = '../data/processed'

# Load model results from NB08
model_file = os.path.join(PROCESSED_DIR, 'model_comparison.csv')
if os.path.exists(model_file):
    model_results = pd.read_csv(model_file, index_col=0)
    print('✅ Model comparison loaded')
    print(model_results)
else:
    model_results = None
    print('⚠️ model_comparison.csv not found — run NB08 first')

# Load feature data
feat_file = os.path.join(PROCESSED_DIR, 'tata_motors_all_features.csv')
clean_file = os.path.join(PROCESSED_DIR, 'tata_motors_clean.csv')
if os.path.exists(feat_file):
    df = pd.read_csv(feat_file, index_col=0, parse_dates=True)
elif os.path.exists(clean_file):
    df = pd.read_csv(clean_file, index_col=0, parse_dates=True)
else:
    df = None
print(f'Price data: {df.shape if df is not None else "Not found"}')


---

## Part 0: Understanding the "Noise Ceiling"

### Why 55% is NOT a coin flip

This is the most common reaction: *"55% accuracy? That's barely better than random!"*

The **50-Year Veteran** says:
> "In the casino, the house edge is only 52%. That 2% built Las Vegas. In the stock market, 55% is a money-printing machine **IF** you size your bets correctly."

The **Data Scientist** says:
> "You are hitting the 'Noise Ceiling.' Financial data is 90% noise and 10% signal. To break past 55%, you don't need a better model — you need **better data** and **smarter targets**."

### The Mathematics of Edge

$$\text{Expected PnL per trade} = (p \times w) - ((1-p) \times l)$$

Where:
- $p$ = win probability (our 55%)
- $w$ = average win size
- $l$ = average loss size

Even with $w = l$ (symmetric payoffs), a 55% edge compounds into massive wealth over hundreds of trades.


In [ ]:
# ============================================================
# 0.1 Demonstrate the power of edge via Monte Carlo simulation
# ============================================================
np.random.seed(42)

# Pull REAL accuracy from NB08 results
if model_results is not None:
    best_model_name = model_results['Accuracy'].idxmax()
    real_accuracy = model_results.loc[best_model_name, 'Accuracy']
    print(f'🏆 Best model from NB08: {best_model_name}')
    print(f'   Real accuracy: {real_accuracy:.4f} ({real_accuracy*100:.2f}%)')
else:
    real_accuracy = 0.55  # Fallback
    print('Using default accuracy: 55%')

# Monte Carlo: Compound edge over 500 trades
n_sims = 1000
n_trades = 500
avg_win = 0.015   # 1.5% avg win
avg_loss = 0.015  # 1.5% avg loss

final_wealth = []
for _ in range(n_sims):
    capital = 100
    for _ in range(n_trades):
        if np.random.random() < real_accuracy:
            capital *= (1 + avg_win)
        else:
            capital *= (1 - avg_loss)
    final_wealth.append(capital)

final_wealth = np.array(final_wealth)

print(f'\nMonte Carlo Simulation ({n_sims} runs × {n_trades} trades):')
print(f'  Starting Capital: ₹100')
print(f'  Win Rate: {real_accuracy*100:.2f}%')
print(f'  Avg Win/Loss: ±{avg_win*100:.1f}%')
print(f'  Median Final Capital: ₹{np.median(final_wealth):.2f}')
print(f'  Mean Final Capital:   ₹{np.mean(final_wealth):.2f}')
print(f'  5th Percentile:       ₹{np.percentile(final_wealth, 5):.2f}')
print(f'  95th Percentile:      ₹{np.percentile(final_wealth, 95):.2f}')
print(f'  % Profitable Runs:    {(final_wealth > 100).mean()*100:.1f}%')


In [ ]:
# Visualize the Monte Carlo
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribution of outcomes
ax = axes[0]
ax.hist(final_wealth, bins=50, color='#3498DB', alpha=0.7, edgecolor='white')
ax.axvline(100, color='red', linestyle='--', linewidth=2, label='Starting Capital (₹100)')
ax.axvline(np.median(final_wealth), color='green', linestyle='--', linewidth=2,
           label=f'Median: ₹{np.median(final_wealth):.0f}')
ax.set_title(f'Monte Carlo: {real_accuracy*100:.1f}% Edge × {n_trades} Trades',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Final Capital (₹)', fontsize=11)
ax.legend(fontsize=10)

# Compare edges: 50%, real accuracy, 60%
ax = axes[1]
for edge, color, label in [(0.50, '#E74C3C', 'Random (50%)'),
                            (real_accuracy, '#3498DB', f'Our Model ({real_accuracy*100:.1f}%)'),
                            (0.60, '#2ECC71', 'Target (60%)')]:
    capitals = [100]
    for _ in range(n_trades):
        if np.random.random() < edge:
            capitals.append(capitals[-1] * (1 + avg_win))
        else:
            capitals.append(capitals[-1] * (1 - avg_loss))
    ax.plot(capitals, color=color, linewidth=2, label=label, alpha=0.8)

ax.set_title('Compounding Edge: Single Run Comparison', fontweight='bold', fontsize=13)
ax.set_xlabel('Trade #', fontsize=11)
ax.set_ylabel('Capital (₹)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---

## Part 1: Level 1 — The "Technique" Upgrade (Smart Fixes)

You are currently using a **Binary Classification** approach (Will price go UP or DOWN tomorrow?). This is the hardest thing to predict because a 0.1% move counts the same as a 5% move.

### 1a. Switch to "Meta-Labeling" (The De Prado Method)

This is the gold standard in modern Quant Finance (pioneered by **Marcos Lopez de Prado**).

| Step | Current Approach | Meta-Labeling Upgrade |
|------|-----------------|----------------------|
| Model A | Single model predicts Buy/Sell | Direction model: "Price going UP?" |
| Model B | — | Sizing model: "Given Model A said UP, probability it's right?" |
| Action | Trade on every signal | Only trade when Model B confidence > threshold |

**How it works:** Model B is trained on the *Success/Failure* of Model A. It learns: "Model A is usually wrong when Volatility is high" → filters those trades.

**Result:** *Frequency* drops, but *accuracy* on executed trades shoots up to 60-65%.


In [ ]:
# ============================================================
# 1a. DEMO: Meta-Labeling on Tata Motors data
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

if df is not None:
    # Prepare features
    temp = df.copy()
    if 'Log_Return' not in temp.columns:
        temp['Log_Return'] = np.log(temp['Close'] / temp['Close'].shift(1))
    temp['Target'] = (temp['Log_Return'].shift(-1) > 0).astype(int)

    exclude = ['Target', 'Regime', 'Cluster', 'Log_Return', 'Simple_Return',
               'Price_Change', 'Gain', 'Loss', 'Avg_Gain', 'Avg_Loss', 'RS']
    feat_cols = [c for c in temp.select_dtypes(include=[np.number]).columns if c not in exclude]
    na_pct = temp[feat_cols].isna().mean()
    feat_cols = na_pct[na_pct < 0.3].index.tolist()

    model_df = temp[feat_cols + ['Target']].dropna()
    X = model_df[feat_cols]
    y = model_df['Target']

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

    tscv = TimeSeriesSplit(n_splits=5)

    # MODEL A: Direction prediction (full signals)
    model_a = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    total_a_correct = 0
    total_a_count = 0
    meta_labels = []

    for train_idx, test_idx in tscv.split(X_scaled):
        model_a.fit(X_scaled.iloc[train_idx], y.iloc[train_idx])
        preds_a = model_a.predict(X_scaled.iloc[test_idx])
        actual = y.iloc[test_idx].values
        total_a_correct += (preds_a == actual).sum()
        total_a_count += len(actual)
        # Create meta-labels: 1 = Model A was correct, 0 = Model A was wrong
        meta_labels.extend((preds_a == actual).astype(int))

    acc_a = total_a_correct / total_a_count
    print(f'MODEL A (Direction Only):')
    print(f'  Total predictions: {total_a_count}')
    print(f'  Accuracy: {acc_a:.4f} ({acc_a*100:.2f}%)')
    print(f'  Correct: {total_a_correct} / {total_a_count}')


In [ ]:
# ============================================================
# 1a (cont). META LABELING: Model B filters Model A
# ============================================================
if df is not None:
    # Collect Model A's predictions and probabilities for the test sets
    all_preds_a = []
    all_probs_a = []
    all_test_X = []
    all_actual = []

    for train_idx, test_idx in tscv.split(X_scaled):
        model_a.fit(X_scaled.iloc[train_idx], y.iloc[train_idx])
        preds = model_a.predict(X_scaled.iloc[test_idx])
        probs = model_a.predict_proba(X_scaled.iloc[test_idx])[:, 1]
        all_preds_a.extend(preds)
        all_probs_a.extend(probs)
        all_test_X.append(X_scaled.iloc[test_idx])
        all_actual.extend(y.iloc[test_idx].values)

    all_preds_a = np.array(all_preds_a)
    all_probs_a = np.array(all_probs_a)
    all_actual = np.array(all_actual)
    meta_labels = (all_preds_a == all_actual).astype(int)

    # Filter: only keep trades where Model A confidence > threshold
    thresholds = np.arange(0.50, 0.75, 0.02)
    results_meta = []

    for threshold in thresholds:
        # High confidence = probability far from 0.5
        confidence = np.abs(all_probs_a - 0.5) * 2  # Scale to [0, 1]
        mask = confidence >= (threshold - 0.5) * 2
        if mask.sum() > 10:
            filtered_preds = all_preds_a[mask]
            filtered_actual = all_actual[mask]
            filtered_acc = (filtered_preds == filtered_actual).mean()
            results_meta.append({
                'Threshold': f'{threshold:.0%}',
                'Trades_Taken': mask.sum(),
                'Trades_Filtered': (~mask).sum(),
                'Accuracy': filtered_acc,
                'Filter_Rate': (~mask).mean()
            })

    meta_df = pd.DataFrame(results_meta)
    print('\nMETA-LABELING RESULTS (Confidence Filtering):')
    print('=' * 70)
    print(meta_df.to_string(index=False))

    # Best filtered accuracy
    best_meta = meta_df.loc[meta_df['Accuracy'].idxmax()]
    print(f'\n🎯 Best Meta-Label Result:')
    print(f'   Threshold: {best_meta["Threshold"]}')
    print(f'   Accuracy:  {best_meta["Accuracy"]:.4f} ({best_meta["Accuracy"]*100:.2f}%)')
    print(f'   Trades:    {int(best_meta["Trades_Taken"])} (filtered {int(best_meta["Trades_Filtered"])})')
    print(f'   Improvement over raw: {(best_meta["Accuracy"] - acc_a)*100:+.2f}pp')


In [ ]:
# Visualize Meta-Labeling impact
if df is not None and len(meta_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Accuracy vs Filter Rate
    ax = axes[0]
    ax.plot(meta_df['Filter_Rate'] * 100, meta_df['Accuracy'] * 100,
            'o-', color='#1565c0', linewidth=2, markersize=8)
    ax.axhline(acc_a * 100, color='red', linestyle='--', alpha=0.7,
               label=f'Raw Model A: {acc_a*100:.1f}%')
    ax.set_xlabel('% Trades Filtered Out', fontsize=12)
    ax.set_ylabel('Accuracy on Remaining Trades (%)', fontsize=12)
    ax.set_title('Meta-Labeling: Accuracy vs Selectivity', fontweight='bold', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    # Trade count vs Accuracy
    ax = axes[1]
    colors_bar = ['#2ECC71' if a > acc_a else '#E74C3C' for a in meta_df['Accuracy']]
    ax.bar(range(len(meta_df)), meta_df['Accuracy'] * 100, color=colors_bar, alpha=0.8, edgecolor='white')
    ax.axhline(acc_a * 100, color='red', linestyle='--', linewidth=2, label=f'Baseline: {acc_a*100:.1f}%')
    ax.set_xticks(range(len(meta_df)))
    ax.set_xticklabels(meta_df['Threshold'], rotation=45)
    ax.set_ylabel('Accuracy (%)', fontsize=11)
    ax.set_title('Accuracy at Each Confidence Threshold', fontweight='bold', fontsize=13)
    ax.legend(fontsize=11)

    plt.suptitle('Meta-Labeling: Trade Less, Win More', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()


---

### 1b. The "Triple Barrier" Method

Stop labeling data based on "Price at Close." That's arbitrary.

**Standard Label:** Did price close higher tomorrow? → Binary, noisy.

**Triple Barrier Label:** Did price hit a **Profit Target** level *before* it hit a **Stop Loss** level within a **time horizon**?

$$\text{Label} = \begin{cases} +1 & \text{if price hits } +\tau \text{ (take profit) first} \\ -1 & \text{if price hits } -\tau \text{ (stop loss) first} \\ 0 & \text{if neither within } T \text{ bars} \end{cases}$$

This teaches the model to find *stable* moves, not lucky closes.


In [ ]:
# ============================================================
# 1b. Triple Barrier Implementation
# ============================================================
if df is not None:
    close = df['Close']
    vol = close.pct_change().rolling(21).std()  # 21-day vol

    # Triple barrier parameters
    pt_multiplier = 1.5   # profit take = 1.5 × daily vol
    sl_multiplier = 1.5   # stop loss = 1.5 × daily vol
    max_holding = 5       # max 5 days

    labels_tb = []
    for i in range(len(close) - max_holding):
        entry = close.iloc[i]
        daily_vol = vol.iloc[i] if not np.isnan(vol.iloc[i]) else 0.02
        pt = entry * (1 + pt_multiplier * daily_vol)
        sl = entry * (1 - sl_multiplier * daily_vol)

        label = 0  # timeout
        for j in range(1, max_holding + 1):
            if i + j >= len(close):
                break
            price = close.iloc[i + j]
            if price >= pt:
                label = 1; break
            elif price <= sl:
                label = -1; break
        labels_tb.append(label)

    labels_tb = np.array(labels_tb)
    print('TRIPLE BARRIER LABELING RESULTS:')
    print(f'  Total samples: {len(labels_tb)}')
    print(f'  +1 (Profit Hit): {(labels_tb == 1).sum()} ({(labels_tb == 1).mean()*100:.1f}%)')
    print(f'  -1 (Stop Hit):   {(labels_tb == -1).sum()} ({(labels_tb == -1).mean()*100:.1f}%)')
    print(f'   0 (Timeout):    {(labels_tb == 0).sum()} ({(labels_tb == 0).mean()*100:.1f}%)')
    print(f'  Win Rate (excl timeouts): {(labels_tb == 1).sum() / max((labels_tb != 0).sum(), 1)*100:.1f}%')


---

## Part 2: Level 2 — The "Data" Upgrade (Fuel Injection)

Your current model runs on **Public Data** (Price, Volume, RSI). Everyone has this.  
To get an edge, you need data that explains *why* the price moves.

### 2a. Derivatives Data (Options Flow)

**Theory:** Smart money (institutions) positions in Options *before* the Stock market moves.

| Feature | What It Captures | Edge |
|---------|-----------------|------|
| **Put/Call Ratio** | Are whales hedging downside? | Leading indicator of crashes |
| **Gamma Exposure (GEX)** | If dealers are "Short Gamma," volatility explodes | Predicts vol regime |
| **Dark Pool Prints** | Large block trades off-exchange | Institutional accumulation/distribution |

### 2b. Micro-Structure Features

**Order Book Imbalance:** If there are 10,000 Buy orders and only 2,000 Sell orders at the current price, the price *must* go up to find liquidity. This is **physics**, not a guess.


In [ ]:
# ============================================================
# 2. Simulated Options Flow Analysis
# (Real options data requires paid API — we demonstrate the concept)
# ============================================================
if df is not None:
    # Use real volume data to create proxy signals
    vol_sma = df['Volume'].rolling(21).mean()
    vol_ratio = df['Volume'] / vol_sma  # Volume surge detection

    # High volume days often correlate with institutional activity
    high_vol_days = vol_ratio > 2.0
    normal_days = ~high_vol_days

    # Next-day returns after high-volume vs normal days
    if 'Log_Return' not in df.columns:
        df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
    next_ret = df['Log_Return'].shift(-1)

    print('VOLUME SURGE ANALYSIS (Proxy for Institutional Flow):')
    print('=' * 60)
    print(f'  High-Volume Days (>2× avg): {high_vol_days.sum()}')
    print(f'  Normal Days:                {normal_days.sum()}')
    print(f'\n  Avg Next-Day Return:')
    print(f'    After High Volume: {next_ret[high_vol_days].mean()*100:+.4f}%')
    print(f'    After Normal Vol:  {next_ret[normal_days].mean()*100:+.4f}%')
    print(f'\n  Volatility of Next-Day Return:')
    print(f'    After High Volume: {next_ret[high_vol_days].std()*100:.4f}%')
    print(f'    After Normal Vol:  {next_ret[normal_days].std()*100:.4f}%')

    # Does high volume predict direction?
    hv_up = (next_ret[high_vol_days] > 0).mean()
    nv_up = (next_ret[normal_days] > 0).mean()
    print(f'\n  P(Up | High Volume):  {hv_up*100:.1f}%')
    print(f'  P(Up | Normal Vol):   {nv_up*100:.1f}%')
    print(f'  Institutional Flow Edge: {(hv_up - nv_up)*100:+.1f}pp')


---

## Part 3: Level 3 — The "Expensive" Upgrade (Alternative Data)

This is how hedge funds get to 60%+. They buy data you don't have.

| Data Source | Application to Tata Motors | Cost |
|-------------|---------------------------|------|
| **Satellite Imagery** | Count finished trucks in factory lots → inventory buildup signal | $5K-50K/mo |
| **Shipping Manifests** | Track JLR shipments from UK to China | $1K-10K/mo |
| **Real-Time Steel Futures** | Input cost leading indicator (not just daily closes) | $500-5K/mo |
| **Credit Card Spending** | Car purchase trends before sales reports | $10K-100K/mo |
| **Google Trends** | "Tata Nexon EV" search volume → demand proxy | Free |


In [ ]:
# ============================================================
# 3. Free Alternative Data: Google Trends Proxy via Search Volume
# We use real price momentum as a proxy
# ============================================================
if df is not None:
    # Sector comparison: Compute real correlations with NIFTY Auto
    import yfinance as yf

    print('Downloading sector data for correlation analysis...')
    sector = yf.download('^CNXAUTO', period='5y', progress=False)
    if hasattr(sector.columns, 'levels'):
        sector.columns = sector.columns.get_level_values(0)

    if not sector.empty and 'Close' in sector.columns:
        sector_ret = sector['Close'].pct_change().dropna()
        stock_ret = df['Close'].pct_change().dropna()

        # Align dates
        common = stock_ret.index.intersection(sector_ret.index)
        if len(common) > 30:
            corr = stock_ret.loc[common].corr(sector_ret.loc[common])
            rolling_corr = stock_ret.loc[common].rolling(60).corr(sector_ret.loc[common])

            print(f'\n  NIFTY Auto Correlation:')
            print(f'    Full-period correlation: {corr:.4f}')
            print(f'    Mean rolling (60d) corr: {rolling_corr.mean():.4f}')
            print(f'    Max rolling corr:        {rolling_corr.max():.4f}')
            print(f'    Min rolling corr:        {rolling_corr.min():.4f}')
            print(f'\n  → Correlation > 0.5 means sector moves explain TMCV well')
            print(f'  → This confirms: "Watch the Sector, Trade the Stock"')
    else:
        print('  Sector data not available')


---

## Part 4: Level 4 — The "Reality Check" (Risk Management)

### The Veteran's Warning:
> *"If you show me a backtest with 70% accuracy, I will show you a model that is overfitting."*

When you try to force accuracy higher, the model starts **memorizing the past**.

### The Better Metric: The Information Coefficient (IC)

$$IC = \text{corr}(\hat{y}, y) = \frac{\text{cov}(\text{prediction}, \text{actual})}{\sigma_{\text{pred}} \cdot \sigma_{\text{actual}}}$$

A consistent IC of 0.05 (tiny!) is **better** than a volatile Accuracy of 60%.


In [ ]:
# ============================================================
# 4. Information Coefficient (IC) Calculation
# ============================================================
if df is not None and model_results is not None:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import TimeSeriesSplit
    from sklearn.preprocessing import StandardScaler

    temp = df.copy()
    if 'Log_Return' not in temp.columns:
        temp['Log_Return'] = np.log(temp['Close'] / temp['Close'].shift(1))
    temp['Target'] = (temp['Log_Return'].shift(-1) > 0).astype(int)

    exclude = ['Target', 'Regime', 'Cluster', 'Log_Return', 'Simple_Return',
               'Price_Change', 'Gain', 'Loss', 'Avg_Gain', 'Avg_Loss', 'RS']
    feat_cols = [c for c in temp.select_dtypes(include=[np.number]).columns if c not in exclude]
    na_pct = temp[feat_cols].isna().mean()
    feat_cols = na_pct[na_pct < 0.3].index.tolist()
    model_clean = temp[feat_cols + ['Target']].dropna()
    X = model_clean[feat_cols]
    y_ic = model_clean['Target']

    scaler = StandardScaler()
    X_sc = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

    tscv = TimeSeriesSplit(n_splits=5)
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

    ic_per_fold = []
    for train_idx, test_idx in tscv.split(X_sc):
        rf.fit(X_sc.iloc[train_idx], y_ic.iloc[train_idx])
        probs = rf.predict_proba(X_sc.iloc[test_idx])[:, 1]
        actual = y_ic.iloc[test_idx].values
        ic = np.corrcoef(probs, actual)[0, 1]
        ic_per_fold.append(ic)

    print('INFORMATION COEFFICIENT (IC) — The Institutional Metric:')
    print('=' * 60)
    for i, ic in enumerate(ic_per_fold):
        bar = '█' * int(abs(ic) * 100)
        print(f'  Fold {i+1}: IC = {ic:+.4f}  {bar}')
    print(f'\n  Mean IC:   {np.mean(ic_per_fold):+.4f}')
    print(f'  Std IC:    {np.std(ic_per_fold):.4f}')
    print(f'  IC Ratio:  {np.mean(ic_per_fold)/max(np.std(ic_per_fold), 1e-6):.2f}')
    print(f'\n  Interpretation:')
    print(f'    IC > 0.02: Model has genuine skill')
    print(f'    IC > 0.05: Strong institutional-grade signal')
    print(f'    IC Ratio > 0.5: Signal is consistent across time')


---

## Part 5: Kelly Criterion — Sizing Your 55% Edge

The **Kelly Criterion** tells you exactly how much to bet given your edge:

$$f^* = \frac{p \cdot b - q}{b}$$

Where:
- $f^*$ = fraction of capital to bet
- $p$ = probability of winning
- $b$ = win/loss ratio (odds)
- $q = 1 - p$ = probability of losing


In [ ]:
# ============================================================
# 5. Kelly Criterion with REAL accuracy
# ============================================================
if model_results is not None:
    best_acc = model_results['Accuracy'].max()
    p = best_acc
    q = 1 - p
    b = 1.0  # Symmetric payoffs (win size = loss size)

    kelly_full = (p * b - q) / b
    kelly_half = kelly_full / 2  # Half-Kelly (conservative)
    kelly_quarter = kelly_full / 4  # Quarter-Kelly (ultra-safe)

    print(f'KELLY CRITERION — Optimal Position Sizing:')
    print(f'=' * 60)
    print(f'  Real Win Rate (from NB08): {p:.4f} ({p*100:.2f}%)')
    print(f'  Win/Loss Ratio:            {b:.1f} (symmetric)')
    print(f'\n  Full Kelly:    {kelly_full*100:.2f}% of capital per trade')
    print(f'  Half Kelly:    {kelly_half*100:.2f}% of capital per trade (recommended)')
    print(f'  Quarter Kelly: {kelly_quarter*100:.2f}% of capital per trade (conservative)')

    # Simulate Kelly compounding
    np.random.seed(42)
    n_trades = 500

    results_kelly = {}
    for name, frac in [('Full Kelly', kelly_full), ('Half Kelly', kelly_half),
                        ('Quarter Kelly', kelly_quarter), ('Fixed 5%', 0.05)]:
        capital = 100
        history = [capital]
        for _ in range(n_trades):
            bet = capital * frac
            if np.random.random() < p:
                capital += bet * b
            else:
                capital -= bet
            history.append(capital)
        results_kelly[name] = history

    print(f'\n  After {n_trades} trades:')
    for name, hist in results_kelly.items():
        print(f'    {name:15s}: ₹{hist[-1]:,.2f} (Return: {(hist[-1]/100-1)*100:+.1f}%)')

    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    colors_k = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']
    for (name, hist), clr in zip(results_kelly.items(), colors_k):
        ax.plot(hist, color=clr, linewidth=2, label=name, alpha=0.8)
    ax.set_title(f'Kelly Criterion: Position Sizing with {p*100:.1f}% Edge',
                 fontweight='bold', fontsize=14)
    ax.set_xlabel('Trade #', fontsize=12)
    ax.set_ylabel('Capital (₹)', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')
    plt.tight_layout()
    plt.show()


---

## Summary Action Plan

### To improve the model RIGHT NOW without buying expensive data:

| # | Action | Expected Impact |
|---|--------|----------------|
| 1 | **Stop predicting "Up/Down"** | Reduces noise in labels |
| 2 | **Predict "Regime"** — Is today safe to trade? | Filters out dangerous environments |
| 3 | **Use Meta-Labeling** — Keep top 20% confidence trades | +5-10pp accuracy on filtered trades |
| 4 | **Accept the 55%** — Use Kelly Criterion sizing | Compounds small edge into wealth |

### Final Verdict:
> **Don't chase 70% accuracy. Chase higher *profitability* on the 55% accuracy you already have.**

---

*Next: Notebook 15 — Transfer Learning (Breaking the 60% Barrier)*
